# RAG Evaluation with RAGAS

**Intro to RAG evaluation · The RAGAS metrics · Preparing a dataset · Evaluating a RAG app**

---

### The question this notebook answers

You built a RAG chatbot. Your boss asks:

> *"Is it good?"*

Right now your honest answer is *"I tried about ten questions and they looked fine."*

That is not an answer. It does not scale, it cannot be repeated, and it cannot tell you whether the change you shipped yesterday made things better or worse.

**RAGAS turns "it looked fine" into a number.**

---

# Part 1 - Introduction to RAG Evaluation and RAGAS

### Why evaluating RAG is harder than normal testing

A normal function has one right answer, so testing is easy:

```
   add(2, 2)  ->  4        correct or not, no argument
```

A RAG app has **many** acceptable answers:

```
   "How long do refunds take?"
      ->  "Refunds take 7 business days."
      ->  "You will get your money back within a week of business days."
      ->  "Seven working days."
```

All three are right. `assert answer == expected` is useless here.

---

### The key insight: a RAG app has TWO halves

This is the most important idea in the notebook.

```
                    HALF 1: RETRIEVER                  HALF 2: GENERATOR
                 (find the right documents)        (write an answer from them)

   question ───────────► search ───────────► documents ───────────► LLM ───────────► answer
                            |                                        |
                            |                                        |
                    can fail by bringing              can fail by ignoring good
                    the wrong documents               documents and making things up
```

**They fail in completely different ways, and the fix is completely different.**

| **What you see**                                    | **Which half broke** | **What you fix**                |
| --------------------------------------------------- | -------------------- | ------------------------------- |
| Answer is wrong, and the documents were wrong too   | Retriever            | chunking, embeddings, search, k |
| Answer is wrong, but the right documents were there | Generator            | the prompt, the model           |

A single "accuracy" score cannot tell these apart. That is exactly the gap RAGAS fills: **it scores each half separately.**

---

### So what is RAGAS?

> **RAGAS is a library that scores your RAG answers from 0 to 1, using an LLM as the judge.**

You give it what your app did (question, documents it found, answer it wrote). RAGAS asks a second LLM careful questions about that, and returns numbers.

```
   your RAG output  ──►  RAGAS  ──►  faithfulness      0.95
                          |          answer relevancy  0.88
                     (judge LLM)     context precision 0.50   <- the weak spot
                                     context recall    0.45
```

### Two things to be clear about up front

1. **The judge is an LLM, so evaluation costs money and takes time.** Every metric on every row is at least one LLM call. 5 questions x 4 metrics is 20-plus calls. Start small.
2. **Scores are not exact truth.** They are a consistent, repeatable opinion. Their real value is **comparing** version A against version B, not the absolute number.

---

# Part 2 - Understanding RAGAS Metrics

### The four you should learn first

| **Metric**            | **The question it asks**                              | **Which half it tests** |
| --------------------- | ----------------------------------------------------- | ----------------------- |
| **Faithfulness**      | Is the answer actually supported by the documents?    | Generator               |
| **Answer relevancy**  | Does the answer address the question that was asked?  | Generator               |
| **Context precision** | Were the retrieved documents useful, or mostly noise? | Retriever               |
| **Context recall**    | Did we retrieve everything needed to answer?          | Retriever               |

All four return **0 to 1, higher is better.**

---

### Reading a low score

This table is the reason you run RAGAS at all.

| **Low score**     | **What it means in plain English**                          | **Where to look**                              |
| ----------------- | ----------------------------------------------------------- | ---------------------------------------------- |
| Faithfulness      | The model is **making things up**, even with good documents | the prompt, the model                          |
| Answer relevancy  | The answer wanders or dodges the question                   | the prompt                                     |
| Context precision | You retrieved a lot of junk alongside the useful bit        | chunk size, k, reranking                       |
| Context recall    | The needed information was **never retrieved**              | chunking, embeddings, the documents themselves |

> **Worked example.** Faithfulness 0.4 with context recall 0.95 means the documents were there and the model ignored them. Do not touch your retriever. Fix the prompt.

---

### The split that decides your whole workflow: does it need a reference answer?

A **reference** is a human-written correct answer, also called ground truth. Writing them is slow, so this matters:

| **Metric**                         | **Needs a reference?** |
| ---------------------------------- | ---------------------- |
| Faithfulness                       | **No**                 |
| Answer relevancy                   | **No**                 |
| Context precision (with reference) | Yes                    |
| Context recall                     | Yes                    |
| Factual correctness                | Yes                    |
| Semantic similarity                | Yes                    |

**Why you care:** the two "No" metrics can run on **live production traffic**, where nobody has written a correct answer. The "Yes" metrics need a prepared test set.

So the usual setup is:

```
   test set with references   ->  all metrics       ->  run before each release
   live production traffic    ->  the 2 no-reference metrics  ->  run continuously
```

---

# Part 3 - Preparing a Dataset for RAGAS Evaluation

### Step 1 - Setup

Install the Gemini integration packages once in your notebook environment:

In [1]:
!pip install "numpy>=1.26.2,<2" "ragas==0.3.9" "langchain==0.3.27" "langchain-core==0.3.80" "langchain-community==0.3.31" "langchain-google-genai==2.1.12" "langchain-openai==0.3.35" python-dotenv

  Using cached ragas-0.4.3-py3-none-any.whl.metadata (23 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached scikit_network-0.33.5-cp311-cp311-win_amd64.whl.metadata (4.6 kB)
  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
     ---------------------------------------- 0.0/58.6 kB ? eta -:--:--
     ------ --------------------------------- 10.2/58.6 kB ? eta -:--:--
     -------------------------------------- 58.6/58.6 kB 780.2 kB/s eta 0:00:00
  Using cached langchain_classic-1.0.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached numpy-2.4.6-cp311-cp311-win_amd64.whl.metadata (6.6 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached multiprocess-0.70.19-py311-none-any.whl.metadata (7.5 kB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
google-cloud-aiplatform 1.71.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.36.2 which is incompatible.
matplotlib 3.7.2 requires pyparsing<3.1,>=2.3.1, but you have pyparsing 3.3.2 which is incompatible.
pyautogen 0.7.3 requires websockets<15,>=14, but you have websockets 16.1.1 which is incompatible.
tensorflow-intel 2.15.0 requires numpy<2.0.0,>=1.23.5, but you have numpy 2.4.6 which is incompatible.
tensorflow-intel 2.15.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 7.36.2 which is incompatible.
to

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Add this line to your .env file: GOOGLE_API_KEY=your_gemini_api_key
if os.environ.get("GOOGLE_API_KEY"):
    print("GOOGLE_API_KEY loaded")
else:
    print("GOOGLE_API_KEY MISSING, add it to your .env file")

GOOGLE_API_KEY loaded


```
GOOGLE_API_KEY loaded
```

### Step 2 - The four field names

A RAGAS row has four fields. **The names are strict.**

| **Field**            | **What goes in it**                                    | **Comes from**  |
| -------------------- | ------------------------------------------------------ | --------------- |
| `user_input`         | the question                                           | you             |
| `retrieved_contexts` | the documents your app found, as a **list of strings** | your retriever  |
| `response`           | the answer your app produced                           | your LLM        |
| `reference`          | the correct answer, written by a human                 | you, in advance |

> **This is the number one place RAGAS code breaks.** Older tutorials use the old names. If you copy code from a blog post and get a confusing error, check this table first.
>
> \| Old name (do not use) | Current name | |---|---| | `question` | `user_input` | | `contexts` | `retrieved_contexts` | | `answer` | `response` | | `ground_truth` | `reference` |

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate

GEMINI_MODEL = "gemini-3.8-flash"

documents = [
    "Refunds are processed within 7 business days for orders placed under 30 days ago.",
    "Standard shipping takes 3 to 5 business days. Express shipping takes 1 business day.",
    "Our support team is available Monday to Friday, 9am to 6pm IST.",
    "You can cancel a subscription anytime from Settings. Billing stops at the end of the cycle.",
    "Damaged items must be reported with a photo within 48 hours of delivery.",
]

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = InMemoryVectorStore.from_texts(documents, embedding=embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a support assistant. Answer ONLY using this context: {context}"),
    ("human", "{question}"),
])

print("RAG app ready")

RAG app ready


```
RAG app ready
```

In [7]:
test_questions = [
    "how long do refund take?",
    "How fast is express shipping?",
    "when is the support team available?",
    "How do i cance my subscription"
]

reference_answer = [
    "Refunds are processed within 7 business days.",
    "Express shipping takes 1 business day.",
    "Support is available Monday to Friday, 9am to 6pm IST.",
    "You can cancel anytime from Settings, and billing stops at the end of the cycle.",
]

In [8]:
rows = []

for question, reference in zip(test_questions,reference_answer):

    docs = retriever.invoke(question)

    contexts = []
    for doc in docs:
        contexts.append(doc.page_content)

    context_text = " ".join(contexts)
    messages = prompt.format_messages(context = context_text, question = question)

    answer = llm.invoke(messages).content

    rows.append({
        "user_input": question,
        "retrieved_contexts": contexts,
        "response": answer,
        "reference": reference
    })

print("done")

done


```
done
```

In [9]:
from ragas import EvaluationDataset

dataset = EvaluationDataset.from_list(rows)

print("Dataset is ready: " , len(dataset))

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

```
Dataset is ready:  4
```

In [ ]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import  LangchainEmbeddingsWrapper

judge_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0))
judge_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
)

print("judge ready")

```
judge ready
```

In [ ]:
from ragas.metrics import (
    Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall)

metrics = [
    Faithfulness(), 
    ResponseRelevancy(), 
    LLMContextPrecisionWithReference(), 
    LLMContextRecall()
]

print("done")

```
done
```

In [ ]:
from ragas import  evaluate

result = evaluate(
    dataset = dataset,
    metrics = metrics,
    llm = judge_llm,
    embeddings = judge_embeddings
)

print(result)

```
Evaluating:   6%|▋         | 1/16 [00:02<00:34,  2.30s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  12%|█▎        | 2/16 [00:03<00:23,  1.67s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:  19%|█▉        | 3/16 [00:04<00:18,  1.46s/it]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating: 100%|██████████| 16/16 [00:08<00:00,  1.82it/s]
```

```
{'faithfulness': 1.0000, 'answer_relevancy': 0.8414, 'llm_context_precision_with_reference': 1.0000, 'context_recall': 1.0000}
```

```
```
```

scores = result.to_pandas()

scores

```
|       |                      **user_input** |                            **retrieved_contexts** |                                      **response** |                                     **reference** | **faithfulness** | **answer_relevancy** | **llm_context_precision_with_reference** | **context_recall** |
| ----: | ----------------------------------: | ------------------------------------------------: | ------------------------------------------------: | ------------------------------------------------: | ---------------: | -------------------: | ---------------------------------------: | -----------------: |
| **0** |            how long do refund take? | [Refunds are processed within 7 business days ... | Refunds are processed within 7 business days f... |     Refunds are processed within 7 business days. |              1.0 |             0.676366 |                                      1.0 |                1.0 |
| **1** |       How fast is express shipping? | [Standard shipping takes 3 to 5 business days.... |            Express shipping takes 1 business day. |            Express shipping takes 1 business day. |              1.0 |             0.854157 |                                      1.0 |                1.0 |
| **2** | when is the support team available? | [Our support team is available Monday to Frida... | The support team is available Monday to Friday... | Support is available Monday to Friday, 9am to ... |              1.0 |             0.975550 |                                      1.0 |                1.0 |
| **3** |      How do i cance my subscription | [You can cancel a subscription anytime from Se... | You can cancel your subscription anytime from ... | You can cancel anytime from Settings, and bill... |              1.0 |             0.859364 |                                      1.0 |                1.0 |
```

metric_columns = [ "faithfulness", "answer_relevancy", "llm_context_precision_with_reference", "context_recall"]

print("Average Scores")
print("-" * 50)


for column in metric_columns:
    if column in scores.columns:
        average = scores[column].mean()
        print(f"{column:38} {average:2f}")

```

```

Average Scores
--------------------------------------------------
faithfulness                           1.000000
answer_relevancy                       0.841360
llm_context_precision_with_reference   1.000000
context_recall                         1.000000

```

```

if "faithfulness" in scores.columns:
    worst = scores.sort_values("faithfulness").iloc[0]

    print("worst question :", worst["user_input"])
    print("faithfulness :", round(worst["faithfulness"]))
    print("worst question :", worst["response"])

```

```

worst question : how long do refund take?
faithfulness : 1
worst question : Refunds are processed within 7 business days for orders placed under 30 days ago.
```